Importing all needed libraries and functionalities:

In [1]:
import pandas as pd
from PIL import Image
import imagehash
import os

Data Pipelining Process:
The data used for training is composed of 3 datasets (found online, listed in acknowledgements). However, these datasets all have different formats, labels, and also contain some duplicate images. Thus, after downloading all the datasets, relabelling the label csv files to match their corresponding dataset sources, and putting everything into one big file, we plan to perform the following process to clean the entire dataset:

1. Combine classes, images, and label files into one big file
2. Examine image files to determine which are duplicates, and also check if such images have different class labelings.
3. Drop duplicate images, removing them from the csvfile and fixing differences in labels between the same images (e.x. if one duplicated image has saffron, but another does not.)
4. Examine all labels and determine if labels are too specific, or too similar to others (i.e. chicken vs chicken breast) 
5. Find the number of times each class is mentioned in training/validation/testing data and determine which classes are the most important, and drop classes which are the least important.

In [2]:
train_csv1 = pd.read_csv('Food Classification/All_csv/_classes_fridge_object_train.csv')
train_csv2 = pd.read_csv('Food Classification/All_csv/_classes_fridge_ingredients_train.csv')
train_csv3 = pd.read_csv('Food Classification/All_csv/_classes_smart_refrigerator_train.csv')
train_csv4 = pd.read_csv('Food Classification/All_csv/_classes_fridge_dataset_train.csv')

valid_csv1 = pd.read_csv('Food Classification/All_csv/_classes_fridge_object_valid.csv')
valid_csv2 = pd.read_csv('Food Classification/All_csv/_classes_fridge_ingredients_valid.csv')
valid_csv3 = pd.read_csv('Food Classification/All_csv/_classes_smart_refrigerator_valid.csv')

test_csv1 = pd.read_csv('Food Classification/All_csv/_classes_fridge_object_test.csv')
test_csv2 = pd.read_csv('Food Classification/All_csv/_classes_fridge_ingredients_test.csv')
test_csv3 = pd.read_csv('Food Classification/All_csv/_classes_smart_refrigerator_test.csv')

# read all the csv files 

train_total = pd.concat([train_csv1,train_csv2,train_csv3,train_csv4],ignore_index=True).fillna(0)
valid_total = pd.concat([valid_csv1,valid_csv2,valid_csv3],ignore_index=True).fillna(0)
test_total = pd.concat([test_csv1,test_csv2,test_csv3],ignore_index=True).fillna(0)
all_total = pd.concat([train_total,valid_total,test_total],ignore_index=True).fillna(0)

print(all_total.shape)

(16479, 88)


From the shape of the dataframe, it is determined that there are 16479 images, and 87 class labels. By inspecting the datasets, it can be seen that there are many duplicate images, as well as alterations. Thus we keep the alterations, and remove all duplicates.

In [ ]:
hashes = []
hashes_str = []
duplicates = set()
included = False
img_names = list(all_total['filename'])
for img in img_names:
    hash_name = imagehash.phash(Image.open(f"Food Classification/All/{img}").convert('RGB')) 
    # convert images to RGB to normalize images, make hash for each image 
    hashes.append(hash_name)
    hashes_str.append(str(hash_name))
    # append each hash to a list

for x, hash_original in enumerate(hashes):
    for y, hash_duplicate in enumerate(hashes[x+1:]): # only check images after the current one checked 
        if hash_original - hash_duplicate < 5: # threshold for hamming distance (count of bit positions where hashes differ) is 5
            if hash_duplicate not in duplicates:
                duplicates.add(str(hash_duplicate))
                # only add if not already counted, use sets instead of lists or otherwise as faster 
print("Success!") # print the set of duplicates

We then remake the dataframe, keeping only rows without duplicate hashes. Afterwards, all duplicate images are removed from the dataset. This lets us see that there are 9184 images after removing duplicates!

In [ ]:
all_total['phash'] = hashes_str # add hashes as a new column to the dataframe
all_total = all_total.reset_index(drop=True)

keep_images = list(all_total['filename'])

for i, img in enumerate(keep_images):
    if all_total['phash'][i] in duplicates:
        os.remove(f"Food Classification/All/{img}")

all_total = all_total[~all_total['phash'].isin(duplicates)] # keep only rows without duplicates
print(all_total.shape)

NameError: name 'hashes_str' is not defined

 Afterwards, summary data is calculated, and placed into a csv for inspection. From this, we notice that there are a lot of "duplicated" labels: chicken and chicken breast, strawberry and strawberries, etc. 

In [ ]:
sum_all_unique = pd.DataFrame(all_total.drop(columns = ['filename','phash']).sum())
sum_all_unique = sum_all_unique.sort_values(by = sum_all_unique.columns[0],ascending = True)

print(sum_all_unique.head())
sum_all_unique.to_csv("combined_summary.csv")

                   0
fish             2.0
Orange          13.0
chilli          14.0
red apple       18.0
 Ladies Finger  20.0


We can then combine all the "duplicated" columns into one, and also remove some classes, with a guideline for such being less than 50 images per class.

In [ ]:
all_total['Apple_unique'] = all_total[['red apple', 'green apple', 'Normal_Apple',
                               'Blotch_Apple', 'damaged_apple','Apple','apple']].max(axis=1)
all_total['Beans_unique'] = all_total[[' Beans','green_beans','bean']].max(axis=1)
all_total['Beef_unique'] = all_total[['beef','ground_beef']].max(axis=1)
all_total['Blueberry_unique'] = all_total[['blueberry','blueberries']].max(axis=1)
all_total['Cabbage_unique'] = all_total[[' Cabbage','cabbage']].max(axis=1)
all_total['Carrot_unique'] = all_total[[' Carrot','carrot']].max(axis=1)
all_total['Chicken_unique'] = all_total[['chicken','chicken breast','chicken_breast']].max(axis=1)
all_total['Chilli_unique'] = all_total[[' Green Chilly','chilli']].max(axis=1)
all_total['Cucumber_unique'] = all_total[[' Cucumber','cucumber']].max(axis=1)
all_total['Eggs_unique'] = all_total[['eggs','egg',' Egg']].max(axis=1)
all_total['Eggplant_unique'] = all_total[[' Eggplant','eggplant',' Brinjal']].max(axis=1)
all_total['Ginger_unique'] = all_total[['ginger',' Ginger']].max(axis=1)
all_total['Mushroom_unique'] = all_total[['mushrooms','mushroom']].max(axis=1)
all_total['Orange_unique'] = all_total[['Orange','orange']].max(axis=1)
all_total['Pepper_unique'] = all_total[[' Capsicum','pepper']].max(axis=1)
all_total['Strawberry_unique'] = all_total[['strawberries','strawberry']].max(axis=1)
all_total['Sweet_potato_unique'] = all_total[['sweet_potato','sweet potato']].max(axis=1)
all_total['Tomato_unique'] = all_total[['tomato',' Tomato']].max(axis=1)
all_total['Cheese_unique'] = all_total[['cheese','goat_cheese']].max(axis=1)
all_total['Ladies Finger_unique'] = all_total[' Ladies Finger']
all_total['Chayote_unique'] = all_total[' Chayote']
all_total = all_total.drop(columns=['red apple', 'green apple', 'Normal_Apple',
                            'Blotch_Apple', 'damaged_apple','Apple','apple',' Beans',
                            'green_beans','bean','beef','ground_beef','blueberry','blueberries',
                            ' Cabbage','cabbage', ' Carrot','carrot','chicken','chicken breast',
                            'chicken_breast',' Green Chilly','chilli',' Cucumber','cucumber',
                            'eggs','egg',' Egg',' Eggplant','eggplant',' Brinjal','ginger',
                            ' Ginger','mushrooms','mushroom','Orange','orange',' Capsicum',
                            'pepper','strawberries','strawberry','sweet_potato','sweet potato',
                            'tomato',' Tomato','cheese','goat_cheese',' Ladies Finger',' Chayote','phash'])
all_total = all_total.rename(columns={'Apple_unique':'Apple',
                       'Beans_unique':'Beans',
                       'Beef_unique':'Beef',
                       'Blueberry_unique':'Blueberry',
                       'Cabbage_unique':'Cabbage',
                       'Carrot_unique':'Carrot',
                       'Chicken_unique':'Chicken',
                       'Chilli_unique':'Chilli',
                       'Cucumber_unique':'Cucumber',
                       'Eggs_unique':'Eggs',
                       'Eggplant_unique':'Eggplant',
                       'Ginger_unique':'Ginger',
                       'Mushroom_unique':'Mushroom',
                       'Orange_unique':'Orange',
                       'Pepper_unique':'Pepper',
                       'Strawberry_unique':'Strawberry',
                       'Sweet_potato_unique':'Sweet_potato',
                       'Tomato_unique':'Tomato',
                       'Cheese_unique':'Cheese',
                       'Ladies Finger_unique':'Ladies Finger',
                       'Chayote_unique':'Chayote'})
all_total = all_total.drop(['phash'],axis=1)
print("Success!")
sum_all = pd.DataFrame(all_total.drop(columns = ['filename']).sum())
sum_all = sum_all.sort_values(by = sum_all.columns[0],ascending = True)
print(sum_all.head())


Success!
                  0
fish            2.0
Ladies Finger  20.0
cauliflower    54.0
radish         62.0
pea            65.0


From the dataframe it is seen that fish and ladies fingers are in too little images to be kept. Thus they are removed.

In [ ]:
#all_total = all_total.drop(columns=['fish','Ladies Finger'])
all_total.to_csv("Clean_Data.csv")
print("Success!")

Success!
